# Projeto 4 — Clipping comparativo (peso 2)

**Pergunta:** o que este recorte de notícias da Agência Brasil e da CNN Brasil mostra sobre a cobertura de Lula e Flávio Bolsonaro? Transforme textos em dados auditáveis e escreva uma conclusão proporcional ao tamanho da amostra. A base tem 40 matérias selecionadas, dez por combinação de veículo e foco, publicadas em outubro de 2026. **Não é uma amostra aleatória de toda a cobertura.**

Entregue uma pasta `23-projeto-4-noticias/` no seu repositório privado com notebook executado, `README.md` e tabelas de resultado pequenas. O projeto integra Aulas 21–23 e vale peso 2. Crie sua **própria chave Groq**; nunca a salve nem a envie.

## 0. Carregar e conferir a base

Abra no Colab. Se este notebook estiver numa pasta isolada, ele baixa a tabela publicada no repositório da disciplina. Confira 40 linhas, datas, URLs e o equilíbrio 10×4.

In [ ]:
import pandas as pd
from pathlib import Path
arquivo_local = Path("dados/noticias_lula_flavio_2026.csv")
if not arquivo_local.exists():
    arquivo_local = Path("../dados/noticias_lula_flavio_2026.csv")
url_base = "https://raw.githubusercontent.com/mateuspestana/extracao_analise_2026/main/aulas/23-extracao-busca-semantica/dados/noticias_lula_flavio_2026.csv"
politica = pd.read_csv(arquivo_local if arquivo_local.exists() else url_base)
print(politica.shape)
print(pd.crosstab(politica["veiculo"], politica["foco_amostral"]))
assert len(politica) == 40

(40, 8)
foco_amostral   Flávio Bolsonaro  Lula
veiculo                               
Agência Brasil                10    10
CNN Brasil                    10    10


## 1. Preparação textual — Aula 21

Preserve `texto_analise`; crie `texto_preparado` com regras justificadas. Conte documentos vazios, duplicatas, tokens antes/depois. Mostre três casos lado a lado; um deve conter nome próprio ou negação.

In [ ]:
# Seu código e comentários aqui.
print("Vazios:", politica["texto_analise"].str.strip().eq("").sum())
print("Duplicatas exatas:", politica["texto_analise"].duplicated().sum())
print("Duplicatas de URL:", politica["url"].duplicated().sum())
print("Datas:", politica["data_publicacao"].min(), 'a', politica["data_publicacao"].max())
print("Mediana de caracteres:", politica["texto_analise"].str.len().median())

Vazios: 0
Duplicatas exatas: 0
Duplicatas de URL: 0
Datas: 2026-10-03 a 2026-10-13
Mediana de caracteres: 203.5


In [ ]:
import unicodedata
import re

def sem_acentos(texto):
    partes = unicodedata.normalize("NFD", texto)
    return "".join(letra for letra in partes if unicodedata.category(letra) != "Mn")

def preparar(texto, remover_acentos=False, remover_urls=True):
    texto = str(texto).lower()
    if remover_urls:
        texto = re.sub(r"https?://\S+", " ", texto)
    if remover_acentos:
        texto = sem_acentos(texto)
    tokens = re.findall(r"\b[^\W\d_]+\b", texto, flags=re.UNICODE)
    return " ".join(tokens)

politica["texto_preparado"] = politica["texto_analise"].map(preparar)
politica[["id", "veiculo", "texto_preparado"]].head(3)

,id,veiculo,texto_preparado
0,P01,Agência Brasil,lula defende respeito às religiões no dia de n...
1,P02,Agência Brasil,ato pela reeleição de lula leva milhares à ave...
2,P03,Agência Brasil,meta alega problemas técnicos para menor alcan...


In [ ]:
stopwords = "a o e de da do das dos em no na nos nas um uma para por com que se os as ao foi ser são mais também lula flávio bolsonaro".split()
stopwords = [unicodedata.normalize("NFD", w).encode("ascii", "ignore").decode("ascii") for w in stopwords]
politica["sem_stopwords"] = politica["texto_preparado"].fillna("").map(
    lambda s: " ".join(t for t in s.split() if sem_acentos(t) not in stopwords)
)

frase = "O candidato não apoiou a proposta da escola"
print("Original:", frase)
print("Preparada:", preparar(frase))
print("Sem stopwords:", " ".join(t for t in preparar(frase).split() if sem_acentos(t) not in stopwords))

Original: O candidato não apoiou a proposta da escola
Preparada: o candidato não apoiou a proposta da escola
Sem stopwords: candidato não apoiou proposta escola


In [ ]:
for id_ in ['P02', 'P05', 'P27']:
    i = politica.index[politica["id"] == id_][0]
    print('ID:', id_)
    print("ANTES:", politica.loc[i, "texto_analise"])
    print("DEPOIS:", politica.loc[i, "texto_preparado"])
    print("SEM STOPWORDS:", politica.loc[i, "sem_stopwords"], "\n")

ID: P02
ANTES: Ato pela reeleição de Lula leva milhares à Avenida Paulista. Candidato à reeleição disse que a semana será decisiva, por causa da votação do fim da escala 6x1, agendada para a próxima quarta-feira (14).
DEPOIS: ato pela reeleição de lula leva milhares à avenida paulista candidato à reeleição disse que a semana será decisiva por causa da votação do fim da escala agendada para a próxima quarta feira
SEM STOPWORDS: ato pela reeleição leva milhares avenida paulista candidato reeleição disse semana será decisiva causa votação fim escala agendada próxima quarta feira 

ID: P05
ANTES: Técnicos do TSE não encontram conteúdo falso no canal Exército do Lula. O setor de combate à desinformação do TSE avaliou os textos de mensagens e os vídeos, a pedido de Nunes Marques, após as decisões conflitantes sobre a liberação do canal.
DEPOIS: técnicos do tse não encontram conteúdo falso no canal exército do lula o setor de combate à desinformação do tse avaliou os textos de mensagens e os 

In [ ]:

politica["tokens_antes"] = politica["texto_analise"].str.split().str.len()
politica["tokens_depois"] = politica["texto_preparado"].str.split().str.len()
politica["tokens_sem_stopwords"] = politica["sem_stopwords"].str.split().str.len()
print(politica[["tokens_antes", "tokens_depois", "tokens_sem_stopwords"]].describe().round(1))


       tokens_antes  tokens_depois  tokens_sem_stopwords
count          40.0           40.0                  40.0
mean           32.0           31.6                  19.8
std             7.1            7.2                   4.5
min            17.0           17.0                  11.0
25%            28.0           27.0                  17.0
50%            32.5           32.0                  20.0
75%            37.0           37.0                  23.2
max            46.0           46.0                  28.0


**REGRAS JUSTIFICADAS:**
> Palavras iguais porém com escritas com minúsculas ou maiúsculas começam a ser o mesmo token

> URLs foram removidas por não indicarem o assunto da matéria

> Os acentos foram mantidos e depois o TF-IDF remove eles.

> Só letras, sem números e pontuações, visto que reduz o ruído nos tópicos.

> Foram removidas palavras específicas, como artigos e preposições, visto que não diferenciam assuntos. Assim como Lula e Flávio Bolsonaro, pois apareciam em todos os tópicos.


## 2. Tópicos — Aula 22

Aplique TF-IDF e NMF às 40 linhas. Use poucos tópicos e `random_state=42`. Mostre palavras e manchetes representativas, dê nomes provisórios e faça uma tabela por veículo e foco. Conte também as menções a Lula e Flávio nos títulos e resumos, sempre com o denominador de cada grupo. Explique ao menos um tópico confuso. As editorias do corpus geral da aula não existem nesta base.

In [ ]:
# Seu código e leitura crítica aqui.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

vetor = TfidfVectorizer(strip_accents="unicode", stop_words=stopwords,
                        min_df=2, max_df=0.85, max_features=1000, ngram_range=(1, 2))
X = vetor.fit_transform(politica["texto_preparado"])
nmf = NMF(n_components=4, init="nndsvda", random_state=42, max_iter=300)
pesos = nmf.fit_transform(X)
termos = vetor.get_feature_names_out()
for i, comp in enumerate(nmf.components_):
    print(i, ", ".join(termos[j] for j in comp.argsort()[-8:][::-1]))
politica["topico"] = pesos.argmax(axis=1)
print(pd.crosstab([politica.veiculo, politica.foco_amostral], politica.topico))


0 turno, segundo, segundo turno, senador, pai, candidato, presidencia, candidato presidencia
1 canal, tse, exercito, canal exercito, eleitoral, sobre, nunes marques, marques
2 ato, rio, apoio, defesa, ato rio, reeleicao, educacao, sociais
3 campanha, casa, presidente, vida, site, oferece, site oferece, pretende
topico                           0  1  2  3
veiculo        foco_amostral               
Agência Brasil Flávio Bolsonaro  1  0  6  3
               Lula              0  3  4  3
CNN Brasil     Flávio Bolsonaro  6  2  1  1
               Lula              2  2  3  3


In [ ]:
for i in range(nmf.n_components):
    posicoes = pesos[:, i].argsort()[-3:][::-1]
    print("\nTópico", i)
    for pos in posicoes:
        print(politica.loc[pos, "id"], politica.loc[pos, "veiculo"], "—", politica.loc[pos, "titulo"][:110])


Tópico 0
P39 CNN Brasil — Flávio: Bolsonaro está satisfeito com 1º turno e deu "dicas" para disputa
P38 CNN Brasil — Flávio buscará orientação de Bolsonaro para reta final, dizem aliados
P37 CNN Brasil — Centrão mira pastas com maior orçamento se Flávio vencer

Tópico 1
P05 Agência Brasil — Técnicos do TSE não encontram conteúdo falso no canal Exército do Lula
P27 CNN Brasil — Canal "Exército do Lula" é registrado junto ao TSE
P09 Agência Brasil — Campanha registra canal Exército do Lula na Justiça Eleitoral

Tópico 2
P04 Agência Brasil — Estudantes fazem ato no Rio de Janeiro em defesa de voto em Lula
P15 Agência Brasil — Flávio Bolsonaro faz ato no Rio ao lado de Douglas Ruas
P06 Agência Brasil — Milhares ocupam Copacabana em ato pela democracia e em apoio a Lula

Tópico 3
P26 CNN Brasil — Lula admitirá alto custo de vida e gravará campanha em casa de apoiadores
P01 Agência Brasil — Lula defende respeito às religiões no Dia de Nossa Senhora Aparecida
P16 Agência Brasil — Site oferec

In [ ]:
politica["mencoes_lula"] = politica["texto_analise"].str.count(r"(?i)\bLula\b")
politica["mencoes_flavio"] = politica["texto_analise"].str.count(r"(?i)\bFlávio\b")
politica["cita_lula"] = politica["texto_analise"].str.contains(r"(?i)\bLula\b")
politica["cita_flavio"] = politica["texto_analise"].str.contains(r"(?i)\bFlávio\b")
print("Denominador (matérias por grupo):")
print(pd.crosstab(politica["veiculo"], politica["foco_amostral"]))
print(politica.groupby(["veiculo", "foco_amostral"])[["mencoes_lula", "mencoes_flavio", "cita_lula", "cita_flavio"]].sum())
politica.groupby(["veiculo", "foco_amostral"])[["mencoes_lula", "mencoes_flavio", "cita_lula", "cita_flavio"]].sum().to_csv("mencoes_por_grupo.csv")

#IA UTILIZADA NESSE CÓDIGO

Denominador (matérias por grupo):
foco_amostral   Flávio Bolsonaro  Lula
veiculo                               
Agência Brasil                10    10
CNN Brasil                    10    10
                                 mencoes_lula  mencoes_flavio  cita_lula  \
veiculo        foco_amostral                                               
Agência Brasil Flávio Bolsonaro             6              13          5   
               Lula                        13               4         10   
CNN Brasil     Flávio Bolsonaro             6              11          5   
               Lula                        10               2         10   

                                 cita_flavio  
veiculo        foco_amostral                  
Agência Brasil Flávio Bolsonaro           10  
               Lula                        3  
CNN Brasil     Flávio Bolsonaro           10  
               Lula                        2  


**NOMES PROVISÓRIOS E TÓPICO CONFUSO:**
> Tópico 0: Segundo turno e bastidores da campnha de Flávio

> Tópico 1: Exército do Lula no TSE

> Tópico 2: Atos de campanha na rua e mobilização

> Tópico 3: Mistura sem temas em comum

> O Tópico 3 é o mais confuso, pois junta assuntos diversos ("Minha Casa, Minha Vida", "custo de vida" e "site de criptomoedas") apenas porque compartilham palavras como "casa", "vida e "site". Isso acontece justamente por os textos serem curtos e usarem poucas palavras, assim o modelo agrupa matérias diferentes.

## 3. Extração estruturada — Aula 23

Crie sua própria chave Groq. Reaproveite o esquema e a função `extrair()` da aula principal. Selecione **4 notícias por combinação de veículo e foco** com `groupby(...).sample(n=4, random_state=42)`; são 16 chamadas ao todo. Faça primeiro **uma** chamada de teste. Depois rode o lote **uma vez** e guarde a tabela. Não repita o lote ao ajustar gráficos ou texto do relatório.

Confira se `evidencia` é um trecho literal de `texto_analise`. Audite manualmente pelo menos quatro fichas, uma por combinação. Marque ausências, exageros e atribuições ambíguas; não conserte silenciosamente a saída do modelo.

In [ ]:
from getpass import getpass
chave_groq = getpass("Sua chave Groq (não aparece): ")
amostra_ia = politica.groupby(["veiculo", "foco_amostral"], group_keys=False).sample(n=4, random_state=42)
print(pd.crosstab(amostra_ia.veiculo, amostra_ia.foco_amostral))


Sua chave Groq (não aparece): ··········
foco_amostral   Flávio Bolsonaro  Lula
veiculo                               
Agência Brasil                 4     4
CNN Brasil                     4     4


In [ ]:
import requests, json
modelo_groq = "openai/gpt-oss-20b"
esquema = {
    "type": "object", "additionalProperties": False,
    "properties": {
        "pessoa_central": {"type": ["string", "null"]},
        "fato": {"type": "string"},
        "tema": {"type": "string"},
        "evidencia": {"type": "string"},
        "limite": {"type": "string"}
    },
    "required": ["pessoa_central", "fato", "tema", "evidencia", "limite"]
}

def extrair(texto):
    resposta = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {chave_groq}"},
        json={"model": modelo_groq, "temperature": 0, "reasoning_effort": "low", "max_completion_tokens": 600,
              "messages": [{"role": "system", "content": "Extraia apenas fatos presentes no texto recebido. A evidencia deve ser um trecho literal e contínuo do texto. Se não houver pessoa central, use null. Não infira intenção política nem viés do veículo."},
                           {"role": "user", "content": texto}],
              "response_format": {"type": "json_schema", "json_schema": {"name": "ficha_noticia", "strict": True, "schema": esquema}}},
        timeout=40)
    if resposta.status_code == 429:
        raise RuntimeError("Limite Groq atingido (429). Aguarde antes de tentar outra chamada.")
    resposta.raise_for_status()
    return json.loads(resposta.json()["choices"][0]["message"]["content"])


In [ ]:
# Teste uma notícia antes de rodar o lote.
teste = extrair(amostra_ia.iloc[0]["texto_analise"])
print(teste)
print("Evidência literal:", teste["evidencia"] in amostra_ia.iloc[0]["texto_analise"])


{'pessoa_central': 'Flávio', 'fato': 'TSE manda suspender site que oferece criptomoedas por apoio a Flávio', 'tema': 'eleições', 'evidencia': 'TSE manda suspender site que oferece criptomoedas por apoio a Flávio', 'limite': 'null'}
Evidência literal: True


In [ ]:
# Execute uma vez após revisar a função e a chamada de teste:
fichas = []
for _, linha in amostra_ia.iterrows():
     resultado = extrair(linha["texto_analise"])
     fichas.append({"id": linha["id"], "veiculo": linha["veiculo"],
                   "foco_amostral": linha["foco_amostral"], **resultado,
                   "evidencia_literal": resultado["evidencia"] in linha["texto_analise"]})
fichas = pd.DataFrame(fichas)
fichas.to_csv("fichas_extraidas.csv", index=False)

In [ ]:
print("Evidência literal:", fichas["evidencia_literal"].sum(), "de", len(fichas))
for id_ in ["P16", "P09", "P33", "P23"]:
    i = politica.index[politica["id"] == id_][0]
    ficha = fichas[fichas["id"] == id_].iloc[0]
    print("ID:", id_)
    print("TEXTO:", politica.loc[i, "texto_analise"])
    print("FICHA:", ficha[["pessoa_central", "fato", "tema", "evidencia", "limite"]].to_dict())
    print("Evidência literal:", ficha["evidencia_literal"], "\n")

Evidência literal: 16 de 16
ID: P16
TEXTO: Site oferece criptomoedas por posts de apoio a Flávio Bolsonaro. Campanha nega envolvimento e coligação de Lula denuncia esquema à PGE. Pesquisador avalia que a ação tem ilegalidade.
FICHA: {'pessoa_central': 'Flávio Bolsonaro', 'fato': 'Site oferece criptomoedas por posts de apoio a Flávio Bolsonaro', 'tema': 'Criptomoedas e apoio político', 'evidencia': 'Site oferece criptomoedas por posts de apoio a Flávio Bolsonaro', 'limite': 'null'}
Evidência literal: True 

ID: P09
TEXTO: Campanha registra canal Exército do Lula na Justiça Eleitoral. Mesmo com o registro, Canal deve aguardar 48 horas para retomar publicações no WhatsApp.
FICHA: {'pessoa_central': None, 'fato': 'Campanha registra canal Exército do Lula na Justiça Eleitoral.', 'tema': 'registro de canal na Justiça Eleitoral', 'evidencia': 'Campanha registra canal Exército do Lula na Justiça Eleitoral.', 'limite': '48 horas'}
Evidência literal: True 

ID: P33
TEXTO: O que sabemos sobre a p

**AUDITORIA DAS FICHAS:**
> P16 (Agência Brasil x Flávio): ausência. o "fato" omite que a campanha nega envolvimento e a denúncia à PGE. "limite" veio como texto "null". Atribuição ambígua: a "pessoa_central" é o Flávio, mas quem age é o site.

> P09 (Agência Brasil x Lula): ausência. possui "limite" igual a "48 horas" sem explicar a que se refere.

> P33 (CNN x Flávio): ausência. omite que TSE e entidades atribuíram o problema a falha técnica. O "limite" não é um limite.

> P23 (CNN x Lula): exagero. a "evidência" é o texto inteiro, não um trecho. A "pessoal_central" veio como texto "null".

## 4. Busca — Aula 23

Faça uma pergunta substantiva. Compare top 3 de busca por palavra e de busca semântica nas 40 matérias. Registre IDs, títulos, links e uma avaliação manual da relevância. A consulta precisa poder ser repetida.

In [ ]:
# Seu código e comparação aqui.
pergunta = "Quais notícias tratam de apoio de lideranças políticas a candidatos?"
consulta_palavras = vetor.transform([pergunta.lower()])
ordem_palavras = (X @ consulta_palavras.T).toarray().ravel().argsort()[-3:][::-1]
for pos in ordem_palavras:
    linha = politica.iloc[pos]
    print(linha["id"], linha["titulo"], linha["url"])

P21 MDB em dez estados do Norte e Nordeste declaram apoio à reeleição de Lula https://www.cnnbrasil.com.br/eleicoes/mdb-em-dez-estados-do-norte-e-nordeste-declaram-apoio-a-reeleicao-de-lula/
P04 Estudantes fazem ato no Rio de Janeiro em defesa de voto em Lula https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/estudantes-fazem-ato-no-rio-de-janeiro-em-defesa-de-voto-em-lula
P37 Centrão mira pastas com maior orçamento se Flávio vencer https://www.cnnbrasil.com.br/blogs/isabel-mega/eleicoes/centrao-mira-pastas-com-maior-orcamento-se-flavio-vencer/


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import torch
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", dispositivo)
modelo_busca = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=dispositivo)
vetores_noticias = modelo_busca.encode(politica["texto_analise"].tolist(), normalize_embeddings=True)
vetor_pergunta = modelo_busca.encode([pergunta], normalize_embeddings=True)
scores = cosine_similarity(vetor_pergunta, vetores_noticias)[0]
for pos in scores.argsort()[-3:][::-1]:
    linha = politica.iloc[pos]
    print(linha["id"], round(float(scores[pos]), 3), linha["titulo"], linha["url"])

Dispositivo: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

P14 0.543 Flávio Bolsonaro recebe apoio de governadores eleitos e Cury https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/flavio-bolsonaro-recebe-apoios-de-governadores-eleitos-e-cury
P21 0.513 MDB em dez estados do Norte e Nordeste declaram apoio à reeleição de Lula https://www.cnnbrasil.com.br/eleicoes/mdb-em-dez-estados-do-norte-e-nordeste-declaram-apoio-a-reeleicao-de-lula/
P40 0.501 Michelle sinaliza preferência por exercer mandato caso Flávio seja eleito https://www.cnnbrasil.com.br/blogs/gustavo-uribe/eleicoes/michelle-sinaliza-preferencia-por-exercer-mandato-caso-flavio-seja-eleito/


**AVALIAÇÃO MANUAL DAS BUSCAS:**

> P21, P14, P21 são relevantes

> P40 não é relavante (falso positivo)

> P37, P04 são parciais, visto que uma delas fala sobre negociação de cargos, não um apoio declarado, já a segunda utiliza a palavra estudantes, e eles não são necessariamente lideranças.

## 5. Relatório e entrega

No `README.md`, siga esta ordem: objetivo e pergunta; preparação; revisão de 21–22; roteiro de análise; prática aplicada; resultados do desafio; entrega; declaração de uso de IA; checklist. Inclua fonte, janela temporal, regra de seleção, tabela de denominadores, pelo menos dois achados com IDs/URLs, quatro fichas auditadas, tópico confuso, comparação de buscas e limitações. Separe **descrição dos dados** de **interpretação**. Não chame a diferença observada de “viés do jornal” sem um protocolo mais amplo.

Faça um commit de progresso e um de entrega. Prepare-se para explicar uma decisão de limpeza, um tópico, uma ficha e uma busca.

- [ ] 40 linhas: 10 por veículo × foco.
- [ ] Chave pessoal fora dos arquivos entregues.
- [ ] Extração em amostra de 16, com quatro auditorias.
- [ ] Conclusões citam IDs e links e respeitam o recorte.
- [ ] Notebook executa de cima a baixo no Colab.

**README:**

**Pergunta:**
> o que este recorte de notícias da Agência Brasil e da CNN Brasil mostra sobre a cobertura de Lula e Flávio Bolsonaro?

**Objetivo:**
> Transformar textos curtos de notícias em dados auditáveis (tokens, tópicos, contagens de menções, fichas extraídas por modelo de linguagem e buscas) e tirar conclusões proporcionais ao tamanho da amostra.

**Preparação:**
> Fonte: dados/noticias_lula_flavio_2026.csv, publicado no repositório da disciplina.

> Janela temporal: 3 a 13 de outubro de 2026.

> Regra de seleção: 40 matérias escolhidas, 10 por combinação de veículo e foco.

> Campo analisado: texto_analise (título + resumo). Mediana de 203,5 caracteres, cerca de 32 palavras por matéria.

> Denominadores:
        foco_amostral   Flávio Bolsonaro   Lula
veiculo                               
Agência Brasil                10            10
CNN Brasil                    10            10

**Revisão das aulas 21-22:**
> Aula 21: O texto original deve ser preservado e cada regra precsa ser justificada e mostrada em exemplos.

> Aula 22: o TF-IDF dá peso às palavras que distinguem um documento dos outros, e o NMF agrupa essas palavras em poucos tópicos. O nome de cada tópico é uma leitura humana das palavras e machates representativas, e alguns tópicos não formam um tema coerente.

**Roteiro de análise:**
> Carregar a base e conferir 40 linhas, datas, URLs e o equilíbrio 10x4.

> Crias o texto_preparado e a versão sem stopwords. Contar tokens antes e depois. Mostrar três casos lado a lado

> Aplicar TF-IDF + NMF com 4 tópicos (random-state=42). Nomear os tópicos. cruzar com veículo e foco.

> Contar menções a Lula e a Flávio por grupo, sempre com denominador.

> Extrair fichas estrturadas com Groq numa amostra de 16 matérias (4 por grupo) e auditar 4 delas.

> Comparar o top 3 da busca por palavra com busca por semântica para uma pergunta fixa.

**Achados:**
> 1. as matérias sobre Bolsonaro se concentram em tópicos diferentes em cada veículo. Na CNN se concentram no tópico 0 (ex: P37, "Centrão mira pastas com maior orçamento se Flávio vencer" https://www.cnnbrasil.com.br/blogs/isabel-mega/eleicoes/centrao-mira-pastas-com-maior-orcamento-se-flavio-vencer/) e na Agência Brasil no tópico 2.

> 2. apoio político aparece para os dois candidatos, em veículos diferentes. Na Agência Brasil: P14, "Flávio Bolsonaro recebe apoio de governadores eleitos e Cury" (https://agenciabrasil.ebc.com.br/politica/noticia/2026-10/flavio-bolsonaro-recebe-apoios-de-governadores-eleitos-e-cury). Na CNN: P21, "MDB em dez estados do Norte e Nordeste declaram apoio à reeleição de Lula" (https://www.cnnbrasil.com.br/eleicoes/mdb-em-dez-estados-do-norte-e-nordeste-declaram-apoio-a-reeleicao-de-lula/).

**Tópico confuso:**
> O Tópico 3 é o mais confuso, pois junta assuntos diversos ("Minha Casa, Minha Vida", "custo de vida" e "site de criptomoedas") apenas porque compartilham palavras como "casa", "vida e "site". Isso acontece justamente por os textos serem curtos e usarem poucas palavras, assim o modelo agrupa matérias diferentes.

**Fichas auditadas:**
> P16 (Agência Brasil x Flávio): ausência. o "fato" omite que a campanha nega envolvimento e a denúncia à PGE. "limite" veio como texto "null". Atribuição ambígua: a "pessoa_central" é o Flávio, mas quem age é o site.

> P09 (Agência Brasil x Lula): ausência. possui "limite" igual a "48 horas" sem explicar a que se refere.

> P33 (CNN x Flávio): ausência. omite que TSE e entidades atribuíram o problema a falha técnica. O "limite" não é um limite.

> P23 (CNN x Lula): exagero. a "evidência" é o texto inteiro, não um trecho. A "pessoal_central" veio como texto "null".

**Comparação de buscas:**

> P21, P14, P21 são relevantes

> P40 não é relavante (falso positivo)

> P37, P04 são parciais, visto que uma delas fala sobre negociação de cargos, não um apoio declarado, já a segunda utiliza a palavra estudantes, e eles não são necessariamente lideranças.

**Limitações:**
> 40 materias escolhidas, não aleatórias, 10 por grupo.

> Textos curtos (título + resumo), o que deixa os tópicos dependentes de poucas palavras.

> A limpeza remove números (percentuais, 6x1, prazos...)

> A avaliação de relevância das buscas é manual e feita pelos critérios determinados pela pessoa.


**Uso de Ia:** Foi utilizada para corrigir erro na cécula de stopwords que estava vindo igual a "depois". A célula de contagem de menções também foi feita com auxílio de IA.